# 02 — Associative Memory

Walkthrough of `hope.memory.AssociativeMemory` and its three update rules.

Paper references:

- Definition 1 (Eq. 6): associative memory $M^* = \arg\min_M \tilde{L}(M(K); V)$.
- Eq. 18: Hebbian outer-product update $M \leftarrow M + \eta\, v k^\top$.
- Eq. 93: L2-regression gradient $\nabla_M \|Mk - v\|^2 = (Mk - v)k^\top$ — the `delta` rule.
- Eq. 88 with $\alpha = 1$: adds a $k k^\top$ Hebbian decay term — the `oja` variant here.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from hope.memory import AssociativeMemory

tf.random.set_seed(0)
np.random.seed(0)

KEY_DIM = 6
VAL_DIM = 4
N_STEPS = 200

keys = tf.random.normal((N_STEPS, KEY_DIM))
values = tf.random.normal((N_STEPS, VAL_DIM))
print('seeded', N_STEPS, 'random (k, v) pairs')


## Memory norm over time per rule

We feed the same key/value stream to three memories, one per rule, and plot $\|M\|_F$ at each step.

In [ ]:
rules = ['hebbian', 'delta', 'oja']
norms = {r: [] for r in rules}

for r in rules:
    mem = AssociativeMemory(key_dim=KEY_DIM, value_dim=VAL_DIM, rule=r, learning_rate=0.1)
    for t in range(N_STEPS):
        mem.write(keys[t], values[t])
        norms[r].append(float(tf.norm(mem.memory)))

fig, ax = plt.subplots(figsize=(8, 4))
for r in rules:
    ax.plot(norms[r], label=r)
ax.set_xlabel('step')
ax.set_ylabel('||M||_F')
ax.set_title('Associative-memory norm over time per update rule')
ax.legend()
fig.tight_layout()
plt.show()


## Reconstruction error of the delta rule

The delta rule directly descends the L2-regression objective, so $\|Mk_t - v_t\|$ at write time should shrink quickly.

In [ ]:
mem = AssociativeMemory(key_dim=KEY_DIM, value_dim=VAL_DIM, rule='delta', learning_rate=0.5)
errors = []
for t in range(N_STEPS):
    pred_before = mem.retrieve(keys[t])
    errors.append(float(tf.norm(pred_before - values[t])))
    mem.write(keys[t], values[t])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(errors)
ax.set_xlabel('step')
ax.set_ylabel('||Mk_t - v_t||  (before write)')
ax.set_title('Delta rule shrinks reconstruction error')
fig.tight_layout()
plt.show()


Next: stacking these primitives into a Continuum Memory System (notebook 03).